# Pipeline de Unión de *datasets* de registros socioeconímicos

Este *notebook* implementa un flujo completo de extracción, transformación y carga (ETL) para obtener datos históricos anuales de la *Eurostat* (el portal de *datasets* estadísticos de la Unión Europea). El objetivo es consolidar una base de datos anuales sobre diferentes aspectos socioeconómicos de cada capital/país, como pueden ser:

* La pablación anual.
* El PIB (Producto Interior Bruto).
* Parque de vehículos.

En este *pipeline*, tratamos los siguientes problemas:

### El problema de los Nombres (Estandarización)

Eurostat no es consistente con los nombres de cada capital/país. Por ejemplo, para España, a veces se usa "Madrid", otras "ES30" y otras "Comunidad de Madrid".

Por tanto, creamos una función que "limpia" el texto. Si encuentra un código (como AT13:), lo corta y se queda con el nombre. Luego, consulta el *mapeo_sucio* para que todos los archivos hablen el mismo idioma (por ejemplo, que todos digan "Viena" en lugar de "Wien").

### La Técnica del "Melt" (Caso Andorra)

Los datos de Andorra venían en columnas (2013, 2014, 2015...). Esto es un formato de "lectura humana" pero malo para análisis de datos.

Entonces, usamos *melt* para girar la tabla. Las columnas de años se convierten en filas, creando una columna llamada *Años* y otra *OBS_VALUE*. Esto permite unirlo (*merge*) con los datos de *Eurostat* que ya vienen en formato vertical.

### Cálculo de Ratios (Normalización)

Tenemos los vehículos para cada zona por cada 1000 habitantes, excepto para Andorra. Los datos de Andorra se tienen que buscar en su página oficial de estadísticas, y se encuentran para toda la población, no por cada 1000 habitantes.

Por consiguiente, al conocer la población de Andorra anual, calculamos el ratio dividiendo el total de vehículos por la población y multiplicando por 1000 (regla de 3).

### Integración mediante Merges

En lugar de copiar y pegar, usamos la lógica de bases de datos (*merge*). Usamos Capital/País y Años. Para que un dato se pegue a otro, ambos campos deben coincidir exactamente. Por eso la limpieza de nombres del punto 1 es el paso más importante de todo el código.

In [ ]:
# Importación de librerías

import pandas as pd
import numpy as np

In [ ]:
# =============================================================================
# CONFIGURACIÓN DE RUTAS DE ACCESO
# =============================================================================
# Definimos las variables de ruta para centralizar el acceso a los archivos.
# Usamos el prefijo 'r' (raw string) para que Python no interprete las barras invertidas '\' como caracteres de escape.
ruta_caps = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_socieconómicos\población capitales.csv"
ruta_pais = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_socieconómicos\población países sin capital.csv"
ruta_PIB = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_socieconómicos\PIB.csv"
ruta_vehiculos = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_socieconómicos\vehiculos.csv"
ruta_vehiculos_ad = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_socieconómicos\vehiculos_andorra.csv"
ruta_salida = r"C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_socieconómicos\dataset_socieconómico.csv"

In [ ]:
# =============================================================================
# CARGA DE DATOS (LECTURA)
# =============================================================================
# Leemos los CSV especificando:
# - sep=';' -> El delimitador que usan tus archivos.
# - encoding='latin1' -> Necesario para leer caracteres europeos (ñ, acentos, letras nórdicas).
# - encoding='utf-8-sig' -> Específico para el de Andorra para eliminar el carácter BOM invisible al inicio.

df_pob_caps = pd.read_csv(ruta_caps, sep=';', encoding='latin1')
df_pob_pais = pd.read_csv(ruta_pais, sep=';', encoding='latin1')
df_pib = pd.read_csv(ruta_PIB, sep=';', encoding='latin1')
df_veh = pd.read_csv(ruta_vehiculos, sep=';', encoding='latin1')
df_veh_ad = pd.read_csv(ruta_vehiculos_ad, sep=';', encoding='utf-8-sig')

In [ ]:
# =============================================================================
# DICCIONARIO DE MAPEO (ESTANDARIZACIÓN DE NOMBRES)
# =============================================================================
# Eurostat utiliza múltiples nombres para una misma entidad (códigos NUTS, nombres en inglés/local).
# Este diccionario "mapeo_sucio" traduce todas esas variantes a un nombre común para poder hacer 'merges'.
mapeo_sucio = {
    # AT: Austria (Viena)
    "Wien (greater city)": "Viena", "Wien": "Viena",
    # BA: Bosnia
    "BA: Bosnia y Herzegovina": "Bosnia y Herzegovina",
    # BE: Bélgica (Bruselas)
    "Bruxelles/Brussel (greater city)": "Bruselas",
    "Arr. de Bruxelles-Capitale/Arr. Brussel-Hoofdstad": "Bruselas",
    "Région de Bruxelles-Capitale/Brussels Hoofdstedelijk Gewest": "Bruselas",
    # BG: Bulgaria (Sofía)
    "Sofia": "Sofía", "Sofia (stolitsa)": "Sofía", "Yugozapaden": "Sofía",
    # CH: Suiza (Berna)
    "Bern (greater city)": "Berna", "Espace Mittelland": "Berna",
    # CY: Chipre (Nicosia)
    "Lefkosia (greater city)": "Nicosia", "Kýpros": "Nicosia",
    # CZ: Rep. Checa (Praga)
    "Praha": "Praga", "Hlavní m?sto Praha": "Praga",
    # DE: Alemania (Berlín)
    "Berlin": "Berlín",
    # DK: Dinamarca (Copenhague)
    "København (greater city)": "Copenhague", "Byen København": "Copenhague", "Hovedstaden": "Copenhague",
    # EE: Estonia (Tallín)
    "Tallinn": "Tallín", "Põhja-Eesti": "Tallín", "Eesti": "Tallín",
    # ES: España (Madrid)
    "Madrid (greater city)": "Madrid", "Comunidad de Madrid": "Madrid",
    # FI: Finlandia (Helsinki)
    "Helsinki/Helsingfors": "Helsinki", "Helsinki-Uusimaa": "Helsinki",
    # FR: Francia (París)
    "Paris (greater city)": "París", "Paris": "París", "Ile de France": "París",
    # GR: Grecia (Atenas)
    "Athina (greater city)": "Atenas", "Kentrikos Tomeas Athinon": "Atenas", "Attiki": "Atenas",
    # HR: Croacia (Zagreb)
    "Grad Zagreb": "Zagreb",
    # HU: Hungría (Budapest)
    "Budapest (greater city)": "Budapest", "Budapest": "Budapest",
    # IE: Irlanda (Dublín)
    "Dublin (greater city)": "Dublín", "Dublin": "Dublín", "Eastern and Midland": "Dublín",
    # IS: Islandia (Reikiavik)
    "Reykjavik": "Reikiavik", "Ísland": "Reikiavik",
    # IT: Italia (Roma)
    "Lazio": "Roma",
    # LT: Lituania (Vilna)
    "Vilnius": "Vilna", "Vilniaus apskritis": "Vilna", "Sostin?s regionas": "Vilna",
    # LU: Luxemburgo
    "Luxembourg (greater city)": "Luxemburgo", "Luxembourg": "Luxemburgo",
    # LV: Letonia (Riga)
    "R?ga": "Riga", "Latvija": "Riga",
    # MK: Macedonia
    "North Macedonia": "Macedonia del Norte",
    # MT: Malta
    "Greater Valletta": "La Valeta", "Malta": "La Valeta",
    # NL: Holanda (Ámsterdam)
    "Amsterdam (greater city)": "Ámsterdam", "Netherlands": "Ámsterdam",
    # NO: Noruega (Oslo)
    "Oslo (greater city)": "Oslo", "Oslo og Viken": "Oslo",
    # PL: Polonia (Varsovia)
    "Warszawa": "Varsovia", "Miasto Warszawa": "Varsovia", "Warszawski sto?eczny": "Varsovia",
    # PT: Portugal (Lisboa)
    "Lisboa (greater city)": "Lisboa", "Grande Lisboa": "Lisboa", "Grande Lisboa (Portugal)": "Lisboa",
    # RO: Rumanía (Bucarest)
    "Bucuresti": "Bucarest", "Bucure?ti": "Bucarest", "Bucure?ti-Ilfov": "Bucarest",
    # SE: Suecia (Estocolmo)
    "Stockholm (greater city)": "Estocolmo", "Stockholms län": "Estocolmo", "Stockholm": "Estocolmo",
    # SI: Eslovenia (Liubliana)
    "Ljubljana": "Liubliana", "Osrednjeslovenska": "Liubliana", "Zahodna Slovenija": "Liubliana",
    # SK: Eslovaquia (Bratislava)
    "Bratislava (greater city)": "Bratislava", "Bratislavský kraj": "Bratislava"
}

In [ ]:
# =============================================================================
# FUNCIÓN DE TRANSFORMACIÓN (LÓGICA DE LIMPIEZA)
# =============================================================================
def transformar_nombres(df, col):
    """
    Función para limpiar columnas de ubicación.
    1. Copia el DF para evitar advertencias de SettingWithCopy.
    2. Elimina prefijos NUTS (ej. 'ES30: ') buscando el carácter ':'.
    3. Busca el nombre resultante en el diccionario 'mapeo_sucio'.
    """
    df = df.copy()
    
    def limpiar(valor):
        if not isinstance(valor, str): return valor
        valor = valor.strip() # Elimina espacios en blanco al inicio y final
        
        # Lógica para códigos NUTS: Si hay un ':', tomamos solo lo que hay después.
        if ":" in valor:
            valor = valor.split(":", 1)[1].strip()
            
        # Retorna la traducción del diccionario. Si no existe, mantiene el valor original.
        return mapeo_sucio.get(valor, valor)

    df[col] = df[col].apply(limpiar)
    return df

In [ ]:
# =============================================================================
# UNIFICACIÓN DE LA POBLACIÓN (MASTER TABLE)
# =============================================================================
# El objetivo aquí es crear una tabla base con Capital/País y Años.

# Extraemos columnas de Capitales (urb_cpop)
df_caps = df_pob_caps[['cities', 'TIME_PERIOD', 'OBS_VALUE']].copy()
df_caps.columns = ['Capital/País', 'Años', 'Población']

# Extraemos columnas de Países (demo_pjan)
df_pais = df_pob_pais[['geo', 'TIME_PERIOD', 'OBS_VALUE']].copy()
df_pais.columns = ['Capital/País', 'Años', 'Población']

# Unimos (Concatenamos) ambas tablas verticalmente
df_pob = pd.concat([df_caps, df_pais], ignore_index=True)

# Limpiamos los nombres para que coincidan (ej. transformar 'Wien' en 'Viena')
df_pob = transformar_nombres(df_pob, col="Capital/País")
df_pob["Años"] = df_pob["Años"].astype(int) # Aseguramos que el año sea número entero

In [ ]:
# =============================================================================
# TRATAMIENTO DE ANDORRA (CÁLCULO DE VEHÍCULOS)
# =============================================================================
# El archivo de Andorra está en formato 'Ancho' (una columna por año). Lo pasamos a 'Largo'.
df_veh_ad_long = df_veh_ad.melt(id_vars=["Codi", "Descripció"], var_name="Años", value_name="OBS_VALUE")

# Filtramos para quedarnos solo con la categoría de Turismos
df_veh_ad_long = df_veh_ad_long[df_veh_ad_long['Descripció'].str.contains('TURISMES', na=False)]
df_veh_ad_long["Años"] = df_veh_ad_long["Años"].astype(int)

# Cruzamos con la tabla de población para obtener el denominador
df_pob_ad = df_pob[df_pob["Capital/País"] == "Andorra"][["Años", "Población"]]
df_veh_ad_merge = df_veh_ad_long.merge(df_pob_ad, on="Años", how="left")

# Calculamos el ratio: (Coches / Población) * 1000 habitantes
df_veh_ad_merge["Vehiculos_1000_hab"] = round((pd.to_numeric(df_veh_ad_merge["OBS_VALUE"]) / pd.to_numeric(df_veh_ad_merge["Población"])) * 1000)
df_veh_ad_merge["Capital/País"] = "Andorra"

In [ ]:
# =============================================================================
# UNIFICACIÓN DE VEHÍCULOS GLOBAL
# =============================================================================
# Procesamos el archivo de vehículos general (Eurostat)
df_veh_limpio = transformar_nombres(df_veh, col="geo")
df_veh_limpio = df_veh_limpio.rename(columns={"geo": "Capital/País", "TIME_PERIOD": "Años", "OBS_VALUE": "Vehiculos_1000_hab"})

# Concatenamos Andorra con el resto de Europa
df_vehiculos_total = pd.concat([
    df_veh_ad_merge[["Años", "Capital/País", "Vehiculos_1000_hab"]],
    df_veh_limpio[["Años", "Capital/País", "Vehiculos_1000_hab"]]
], ignore_index=True)

In [ ]:
# =============================================================================
# INTEGRACIÓN FINAL (MERGES)
# =============================================================================
# Paso 1: Limpiar el DataFrame de PIB
df_pib_limpio = transformar_nombres(df_pib, col="geo")
df_pib_limpio = df_pib_limpio.rename(columns={"geo": "Capital/País", "TIME_PERIOD": "Años", "OBS_VALUE": "PIB"})

# Paso 2: Unir Población + PIB (Usamos Left Join para no perder filas de población)
df_final = df_pob.merge(df_pib_limpio[["Capital/País", "Años", "PIB"]], on=["Capital/País", "Años"], how="left")

# Paso 3: Unir el resultado anterior + Vehículos
df_final = df_final.merge(df_vehiculos_total, on=["Capital/País", "Años"], how="left")

# Paso 4: Crear la columna de ID de País (ISO) basada en el nombre de la ciudad/país
capital_a_id = {
    "Andorra": "AD", "Albania": "AL", "Viena": "AT", "Bosnia y Herzegovina": "BA",
    "Bruselas": "BE", "Sofía": "BG", "Berna": "CH", "Nicosia": "CY", "Praga": "CZ",
    "Berlín": "DE", "Copenhague": "DK", "Tallín": "EE", "Madrid": "ES", "Helsinki": "FI",
    "París": "FR", "Atenas": "GR", "Zagreb": "HR", "Budapest": "HU", "Dublín": "IE",
    "Reikiavik": "IS", "Roma": "IT", "Vilna": "LT", "Luxemburgo": "LU", "Riga": "LV",
    "Montenegro": "ME", "Macedonia del Norte": "MK", "La Valeta": "MT", "Ámsterdam": "NL",
    "Oslo": "NO", "Varsovia": "PL", "Lisboa": "PT", "Bucarest": "RO", "Serbia": "RS",
    "Estocolmo": "SE", "Liubliana": "SI", "Bratislava": "SK", "Kosovo": "XK"
}
df_final["Pais_id"] = df_final["Capital/País"].map(capital_a_id)

In [ ]:
# =============================================================================
# FINALIZACIÓN Y EXPORTACIÓN
# =============================================================================
# Convertimos todas las columnas numéricas para asegurar que no haya strings. 'coerce' pone NaN si hay error.
cols_num = ["Población", "PIB", "Vehiculos_1000_hab"]
for c in cols_num:
    df_final[c] = pd.to_numeric(df_final[c], errors="coerce")

# Ordenamos por ID de país y año para una lectura lógica.
df_final = df_final.sort_values(by=["Pais_id", "Años"])

# Exportamos a CSV usando ';' como separador y ',' para decimales (formato europeo Excel).
columnas_orden = ["Pais_id", "Capital/País", "Años", "Población", "PIB", "Vehiculos_1000_hab"]
df_final[columnas_orden].to_csv(ruta_salida, sep=';', decimal=',', index=False, encoding='utf-8-sig')

print(f"✅ Proceso completado. Dataset exportado a: {ruta_salida}")

✅ Proceso completado. Dataset exportado a: C:\Users\Juanfran cd\Desktop\Máster_Ciencia_de_Datos_UA\TFM\datasets\archivos_socieconómicos\dataset_socieconómico.csv
